# Basic notebook to train a CytoVI model for the integration of flow cytometry data

In the following notebook we demonstrate the basic use of CytoVI and how a CytoVI model can be trained to integrate full spectrum cytometry data from two distinct batches. We first import fcs files and store them in the commonly used anndata format for single cell analyses, preprocess the data and do QC checks, train a CytoVI model and inspect the batch corrected latent space of CytoVI. Consecutively, we cluster the CytoVI latent space, anotate the clusters and compute batch-corrected expression estimates. Subsequently, common downstream analysis tasks can be performed such as quantification the relative abundance of cell populations between groups of samples, differential expression analysis or trajectory inference. For more advanced applications of CytoVI we refer to the reproducibility repository and the documentation. Advanced tutorials will be published with the official release of CytoVI into scvi-tools.

In [ ]:
import cytovi
import readfcs
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
# read the flow cytometry data of the two batches and store the raw expression values in adata.layers
adata_batch1 = readfcs.read('../data/raw/Spectral flow/Nunez/For Chiquito/Raw_100000/batch1')
adata_batch1.layers['raw'] = adata_batch1.X.copy()

adata_batch2 = readfcs.read('../data/raw/Spectral flow/Nunez/For Chiquito/Raw_100000/batch2')
adata_batch2.layers['raw'] = adata_batch2.X.copy()

In [ ]:
# preprocess the raw flow cytometry data; we first perform a hyperbolic arcsin transformation, followed by feature-wise min-max scaling
# as a starting point we recommend a global scaling factor of 2000 for full spectrum cytometry data, 100 for conventional flow cytometry data and
# 5 for mass cytometry data; the resulting distribution of scaled expresssion can be visualized in scatter plots and histograms (see below)
cytovi.pp.arcsinh(adata_batch1, global_scaling_factor=2000)
cytovi.pp.scale(adata_batch1)

cytovi.pp.arcsinh(adata_batch2, global_scaling_factor=2000)
cytovi.pp.scale(adata_batch2)

In [ ]:
# merge batches
adata = cytovi.pp.merge_batches([adata_batch1, adata_batch2])

In [ ]:
# inspect the expression values in the scaled layer across the different batches
cytovi.pl.histogram(adata, marker = 'all', groupby='batch', layer_key='scaled')
cytovi.pl.biaxial(adata, marker_x = 'CD3', marker_y = 'CD4', color='batch', layer_key='scaled')

In [ ]:
# optional: subsample the anndata in equal proportions for each batch to speed up training for the purpose of this tutorial
adata = cytovi.pp.subsample(adata, groupby='batch', n_obs=10000)

In [ ]:
# register the anndata and train the CytoVI model controlling for the 'batch' covariate
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata)
model.train()

In [ ]:
# optional: save model on disk for later use
model_path = '../saved_models/'
model.save(f'{model_path}my_cytovi_model', overwrite=True)

In [ ]:
# optional: load model from disk
model = cytovi.CytoVI.load(f'{model_path}my_cytovi_model', adata=adata)

In [ ]:
# plot training dynamics, note: here we want to observe a plateau of the ELBO
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')
plt.show()

In [ ]:
# compute umap of the CytoVI latent space and cluster the latent space using leiden
adata.obsm["X_CytoVI"] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep="X_CytoVI")
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="leiden_CytoVI")

In [ ]:
# plot the the integrated CytoVI latent space and the corresponding clusters including their expression profile
sc.pl.umap(adata, color=["leiden_CytoVI", "batch"])
sc.pl.matrixplot(adata, var_names=adata.var_names, groupby="leiden_CytoVI", layer='scaled', dendrogram=True, standard_scale='var')

In [ ]:
# annotate the leiden clusters into cell types and visualize on the CytoVI latent space
annot_dict = {
    '0': 'Granulocytes',
    '1': 'Monocytes',
    '2': 'Monocytes',
    '3': 'B cells',
    '4': 'T cells',
    '5': 'NK cells'
}
adata.obs['cell_type'] = adata.obs['leiden_CytoVI'].map(annot_dict)
sc.pl.umap(adata, color = ['batch', 'leiden_CytoVI', 'cell_type'])

In [ ]:
# get imputed batch-corrected expression; note: by default we return the mean across all batches, you can change this by setting the transform_batch argument
adata.layers['imputed'] = model.get_normalized_expression(adata, n_samples = 10)

# visualize imputed expression
sc.pl.umap(adata, color = ['CD3', 'CD4', 'CD8', 'CD19', 'CD56'], layer='imputed')

In [ ]:
# optional: save the anndata
adata.write('../data/processed/Nunez_100k_annotated.h5ad')